<a href="https://colab.research.google.com/github/hemajuluri/Ethical-and-fairness/blob/Version2_thesis/03_Transition_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install necessary NLP libraries
!pip install ollama textstat vaderSentiment pandas tqdm

import pandas as pd
import numpy as np
from tqdm import tqdm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import textstat
import time

# Note: This assumes you have Ollama running locally or via a tunneling service
# If using a cloud API, replace with the appropriate library (e.g., openai or langchain)
import ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.8 MB/s eta 0:00:00


In [2]:
# Load the artifacts from Notebook 02
from google.colab import drive
drive.mount('/content/drive')
data_path = '/content/drive/MyDrive/Thesis/src/artifacts/shap_attribution_store.csv'
df = pd.read_csv(data_path)

print(f"✅ Loaded {len(df)} samples for GenAI processing.")
df.head()

Mounted at /content/drive
✅ Loaded 496 samples for GenAI processing.


,applicant_id,gender,reason_1,shap_1,value_1,reason_2,shap_2,value_2,reason_3,shap_3,value_3
0,34,F,Years of Employment,5.204409,365243.0,AMT_GOODS_PRICE,1.917941,112500.0000,EXT_SOURCE_3,0.964148,0.149167
1,49,M,FLAG_EMP_PHONE,0.811314,1.0,AMT_GOODS_PRICE,0.723941,369000.0000,ORGANIZATION_TYPE_Construction,0.653050,NaN
2,71,F,AMT_GOODS_PRICE,1.331414,238500.0,APARTMENTS_AVG,1.229722,0.1732,FLAG_EMP_PHONE,0.811314,1.000000
3,80,F,AMT_GOODS_PRICE,1.394257,225000.0,FLAG_EMP_PHONE,0.811314,1.0000,NAME_INCOME_TYPE_Pensioner,0.425059,NaN
4,85,M,FLAG_EMP_PHONE,0.811314,1.0,EXT_SOURCE_3,0.605857,0.2750,NAME_INCOME_TYPE_Pensioner,0.425059,NaN


In [3]:
def expand_shorthand(value, reason):
    """
    Translates technical placeholders into natural language
    to prevent LLM hallucinations.
    """
    if str(value) == '365243.0' or str(value) == '365243':
        return "currently not employed or is a pensioner"

    # Add logic for currency formatting if applicable
    if 'PRICE' in reason or 'AMT' in reason:
        return f"${value:,.2f}"

    return value

# Applying expansion to make the data 'LLM-Ready'
df['clean_value_1'] = df.apply(lambda x: expand_shorthand(x['value_1'], x['reason_1']), axis=1)

In [36]:

def generate_letter(reason, value):
   # STEP 1: Rule-Based Humanizer (Fixes the '20.2' Grade level instantly)
   human_map = {
       'FLAG_EMP_PHONE': 'we could not verify your work phone number',
       'AMT_GOODS_PRICE': 'the price of the item you want to buy is too high for this loan',
       'Years of Employment': 'your total time spent at your current job',
       'EXT_SOURCE_3': 'the score from our outside credit partners'
   }

   clean_reason = human_map.get(reason, reason.replace('_', ' ').lower())


   # STEP 2: The One-Shot System Prompt
   system_instruction = """
   You are a kind and simple bank clerk.

   ONE-SHOT EXAMPLE TO FOLLOW:
   User: Reason: work phone, Value: 1.0
   Assistant: Thank you for your application. We cannot move forward right now because we could not verify your work phone number. We look forward to seeing a new application from you in six months!


   STRICT RULES:
   1. Copy the style of the example above.
   2. Use short, 5th-grade words only.
   3. No technical codes or underscores.
   4. Start with 'Thank you' and end with 'six months'.
   """


   user_content = f"Reason: {clean_reason}, Value: {value}"


   try:
       response = ollama.chat(
           model='mistral',
           messages=[
               {'role': 'system', 'content': system_instruction},
               {'role': 'user', 'content': user_content},
           ],
           options={'temperature': 0.1, 'num_predict': 60} # Low temp for strict following
       )
       return response['message']['content'].strip()
   except Exception as e:
       return f"Error: {str(e)}"

In [37]:
# We run a sample of 5 first to verify, then you can run the full 496
# To run all, replace .head(5) with the full dataframe
results = []

print("🚀 Starting Mistral-7B Inference Loop...")
for index, row in tqdm(df.head(5).iterrows(), total=5):
    letter = generate_letter(row['reason_1'], row['clean_value_1'])
    results.append(letter)

df_results = df.head(5).copy()
df_results['llm_explanation'] = results

🚀 Starting Mistral-7B Inference Loop...


100%|██████████| 5/5 [02:54<00:00, 34.98s/it]


In [39]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

df_results['sentiment_score'] = df_results['llm_explanation'].apply(get_sentiment)
# Summary for Thesis
print(f"📊 Average Empathy (Sentiment) Score: {df_results['sentiment_score'].mean():.2f}")
# (Target: > 0.05 for positive/supportive tone)

📊 Average Empathy (Sentiment) Score: 0.62


### Testing the Updated Empathetic Prompt with a Single Example

In [38]:
sample_row = df.iloc[0]
sample_letter = generate_letter(sample_row['reason_1'], sample_row['clean_value_1'])

print("Generated Letter for Sample Row:")
print(sample_letter)

sample_sentiment_score = get_sentiment(sample_letter)
sample_readability_grade = textstat.flesch_kincaid_grade(sample_letter)

print(f"\nSample Letter Sentiment Score: {sample_sentiment_score:.2f}")
print(f"Sample Letter Readability Grade: {sample_readability_grade:.1f}")

Generated Letter for Sample Row:
Thank you for your application. Unfortunately, we cannot move forward right now because we could not verify the length of time you have been working at your current job. We look forward to seeing a new application from you in six months!

Sample Letter Sentiment Score: 0.10
Sample Letter Readability Grade: 7.0


In [40]:
df_results['readability_grade'] = df_results['llm_explanation'].apply(textstat.flesch_kincaid_grade)

print(f"📊 Average Readability Grade Level: {df_results['readability_grade'].mean():.1f}")
# (Target: Grade 8.0 - 10.0 for general accessibility)

📊 Average Readability Grade Level: 6.6


In [9]:
def hallucination_check(row):
    # Simple check: Does the keyword of the reason appear in the letter?
    keyword = str(row['reason_1']).split('_')[-1].lower()
    return keyword in row['llm_explanation'].lower()

df_results['passed_hallucination_check'] = df_results.apply(hallucination_check, axis=1)

# Save the final artifact for Phase 7 (Fairness Audit)
output_path = '/content/drive/MyDrive/Thesis/src/artifacts/final_genai_explanations.csv'
df_results.to_csv(output_path, index=False)
print(f"✅ Final explanations saved to {output_path}")

✅ Final explanations saved to /content/drive/MyDrive/Thesis/src/artifacts/final_genai_explanations.csv


In [13]:
# 1. Install system-level dependencies (zstd is required for the installer)
!apt-get update
!apt-get install -y zstd

# 2. Install Python libraries
!pip install ollama textstat vaderSentiment

# 3. Download and Install the Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Start the Ollama server in the background
import subprocess
import time
import os

# We use nohup to ensure the server keeps running in the background
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a moment to initialize
time.sleep(10)

# 5. Pull the Mistral model (required before you can generate letters)
!ollama pull mistral

print("✅ Ollama is installed and Mistral is ready!")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,247 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,006 k

In [14]:
import ollama
try:
    response = ollama.list()
    print("Ollama is awake! Available models:", response)
except Exception as e:
    print("Still having trouble:", e)

Ollama is awake! Available models: models=[Model(model='mistral:latest', modified_at=datetime.datetime(2026, 5, 12, 21, 32, 39, 765869, tzinfo=TzInfo(0)), digest='6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf', size=4372824384, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='7.2B', quantization_level='Q4_K_M'))]
